In [12]:
import numpy as np
import cv2

In [13]:
matrix = np.array([
    [
        [1, 2, 3],
        [3, 4, 1],
        [1, 1, 1]
    ],
    [
        [1, 2, 3],
        [3, 2, 5],
        [7, 1, 1]
    ],
    [
        [1, 2, 3],
        [3, -1, 0],
        [-1, 0, 1]
    ]
])

In [14]:
kernel = np.array([
    [
        [1, 0],
        [1, -1]
    ],
    [
        [-1, 1],
        [1, 0]
    ],
    [
        [-1, 1],
        [0, 1]
    ]
])

In [15]:
def convolucion_multicanal(x, k):
    """Convolución 2D válida: x y k con forma (C, H, W). Suma sobre canales."""
    C, H, W = x.shape
    Ck, kh, kw = k.shape
    assert C == Ck, "Mismo número de canales en entrada y kernel"
    oh, ow = H - kh + 1, W - kw + 1
    salida = np.empty((oh, ow))
    for i in range(oh):
        for j in range(ow):
            parche = x[:, i : i + kh, j : j + kw]
            salida[i, j] = np.sum(parche * k)
    return salida


resultado = convolucion_multicanal(matrix, kernel)
print("Forma de la entrada:", matrix.shape)
print("Forma del kernel:", kernel.shape)
print("Matriz resultante (convolución válida, sin padding, stride 1):")
print(resultado)

Forma de la entrada: (3, 3, 3)
Forma del kernel: (3, 2, 2)
Matriz resultante (convolución válida, sin padding, stride 1):
[[ 4.  9.]
 [ 5. 10.]]


In [16]:
# OpenCV: filter2D aplica correlación 2D por canal (mismo anclaje que arriba: esquina sup. izq.).
# Sumamos los canales para igualar convolucion_multicanal.
# Nota: para convolución clásica habría que rotar el kernel 180°; aquí coincidimos con el bucle manual.

C, H, W = matrix.shape
_, kh, kw = kernel.shape
oh, ow = H - kh + 1, W - kw + 1

resultado_cv = np.zeros((oh, ow), dtype=np.float64)
for c in range(C):
    canal_f = matrix[c].astype(np.float64)
    k_f = kernel[c].astype(np.float64)
    filtrado = cv2.filter2D(
        canal_f,
        ddepth=cv2.CV_64F,
        kernel=k_f,
        anchor=(0, 0),
        borderType=cv2.BORDER_CONSTANT,
        delta=0,
    )
    resultado_cv += filtrado[:oh, :ow]

print("Con OpenCV (cv2.filter2D por canal + suma):")
print(resultado_cv)
print("¿Coincide con el manual?", np.allclose(resultado_cv, resultado))

Con OpenCV (cv2.filter2D por canal + suma):
[[ 4.  9.]
 [ 5. 10.]]
¿Coincide con el manual? True


In [17]:
# Keras (TensorFlow): una sola capa Conv2D suma automáticamente los canales de entrada.
import tensorflow as tf

C, H, W = matrix.shape
_, kh, kw = kernel.shape
entrada = np.transpose(matrix, (1, 2, 0)).astype(np.float32)  # (H, W, C) como en Keras
entrada = entrada[np.newaxis, ...]  # (1, H, W, C)

# Pesos Conv2D: (kh, kw, in_channels, filters)
pesos = np.stack([kernel[c] for c in range(C)], axis=-1)[..., np.newaxis].astype(np.float32)

capa = tf.keras.layers.Conv2D(
    filters=1,
    kernel_size=(kh, kw),
    padding="valid",
    strides=1,
    use_bias=False,
    dtype="float32",
)
capa.build((None, H, W, C))
capa.set_weights([pesos])

oh, ow = H - kh + 1, W - kw + 1
resultado_keras = capa(entrada).numpy().reshape(oh, ow)
print("Con Keras Conv2D:")
print(resultado_keras)
print("¿Coincide con el manual?", np.allclose(resultado_keras, resultado))

Con Keras Conv2D:
[[ 4.  9.]
 [ 5. 10.]]
¿Coincide con el manual? True
